# 03 — Solar output

**Code under test:** `fleximorpv2/models/technologies.py`

**Purpose:** Verify irradiance, temperature derating, and Alaska ice/seasonal modifiers.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Temperature derating direction

**Run the next cell** with identical irradiance but two temperatures (0°C vs 35°C).

**Pass if:** colder run produces **more** annual energy (temp coefficient is −0.004/°C).

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.models.technologies import TechnologyModel, ResourceData

config = load_config("alaska")
model = TechnologyModel(config)
n = 8760
irradiance = np.full(n, 200.0)  # W/m²


def solar_energy(temp_c: float) -> float:
    resource = ResourceData(
        wind_speed=np.zeros(n),
        solar_irradiance=irradiance,
        wave_height=np.zeros(n),
        wave_period=np.zeros(n),
        temperature=np.full(n, temp_c),
        timestamps=np.arange(n),
    )
    design = {"wind_capacity": 0.0, "solar_capacity": 1.0, "hydro_capacity": 0.0}
    return model.calculate_performance(design, resource)["annual_energy"]


cold = solar_energy(0.0)
hot = solar_energy(35.0)
print(f"Cold (0°C):  {cold:.0f} MWh/yr")
print(f"Hot (35°C):  {hot:.0f} MWh/yr")
assert cold > hot, "Expected colder site to outperform hot site at same irradiance"
print("PASS temperature derating direction")